# 專題核心模組：每日歷史新聞增量爬蟲與 FinBERT 情緒量化
本筆記本負責每日自動抓取五大權值股（台積電、鴻海、長榮、中鋼、中信金）之最新財經新聞，並透過預訓練之 NLP 模型進行多空情緒打分。
為防範前視偏誤（Look-Ahead Bias），本模組內建 13:30 收盤時間切分規則，確保盤後新聞之情緒遞延至次日生效。

In [1]:
# Cell 1: 安裝並載入必要套件
# 若在 Colab 環境，通常已內建 pandas 與 requests，只需確保 transformers 與 torch 已安裝
!pip install -q transformers torch pandas requests schedule

import requests
import pandas as pd
from datetime import datetime, timedelta
import time
import torch
from transformers import BertTokenizer, BertForSequenceClassification

print("✅ 套件載入完成！")

# 定義目標五檔個股
TARGET_STOCKS = {
    '2330': '台積電',
    '2317': '鴻海',
    '2603': '長榮',
    '2002': '中鋼',
    '2891': '中信金',
}

✅ 套件載入完成！


In [2]:
# Cell 2: 初始化 NLP 語意模型
print("📥 正在從 Hugging Face 載入中文情感預訓練模型 (這可能需要幾十秒)...")

# 這裡先使用開源的中文情感模型作為範例，未來可無縫替換成你們自定義的 FinBERT-Chinese
# 或是你之前微調好的 Llama-3 API
MODEL_NAME = 'IDEA-CCNL/Erlangshen-Roberta-110M-Sentiment'
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = BertForSequenceClassification.from_pretrained(MODEL_NAME)

def get_finbert_score(text):
    """
    將文字輸入 NLP 模型，取得 [-1, 1] 的多空情緒分數
    """
    if not text:
        return 0.0

    # 延續你在 Colab 處理長文本的優良習慣，設定 max_length 避免 OOM
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        # 依照該模型設計，萃取正面與負面機率差距作為最終 [-1, 1] 情緒分數
        score = probs[0][1].item() - probs[0][0].item()

    return round(score, 4)

print("✅ 模型載入完畢，情緒量化引擎準備就緒！")

📥 正在從 Hugging Face 載入中文情感預訓練模型 (這可能需要幾十秒)...


vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/785 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  409MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ 模型載入完畢，情緒量化引擎準備就緒！


In [3]:
# Cell 3: 定義鉅亨網增量爬蟲與資料清理邏輯
def fetch_today_cnyes_news():
    """只抓取「今天」的鉅亨網台股新聞"""
    url = 'https://api.cnyes.com/media/api/v1/newslist/category/tw_stock'
    headers = {'User-Agent': 'Mozilla/5.0'}

    today = datetime.now()
    # 設定抓取範圍為今天的 00:00:00 到現在
    start_ts = int(datetime(today.year, today.month, today.day).timestamp())
    end_ts = int(today.timestamp())

    daily_news = []

    for page in range(1, 10): # 每天新聞通常不會超過 10 頁
        params = {'limit': 30, 'page': page, 'startAt': start_ts, 'endAt': end_ts}
        try:
            res = requests.get(url, headers=headers, params=params, timeout=10)
            if res.status_code != 200: break

            items = res.json().get('items', {}).get('data', [])
            if not items: break

            for item in items:
                daily_news.append({
                    'publish_time': datetime.fromtimestamp(item.get('publishAt')),
                    'title': str(item.get('title') or '').strip(),
                    'summary': str(item.get('summary') or '').strip(),
                })
            time.sleep(0.3)
        except Exception as e:
            print("爬蟲異常:", e)
            break

    return daily_news

def align_trading_date(publish_time):
    """
    13:30 收盤切分規則：若超過下午 1:30 發布，則遞延至次日
    """
    if publish_time.time() > datetime.strptime('13:30:00', '%H:%M:%S').time():
        return (publish_time + timedelta(days=1)).date()
    return publish_time.date()

In [4]:
# Cell 4: 執行每日 Pipeline，產出 LSTM 特徵表
print(f"🚀 開始執行每日增量結算任務: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# 1. 抓取今日新聞
raw_news = fetch_today_cnyes_news()
print(f"📡 成功抓取全市場今日新聞共 {len(raw_news)} 筆，開始進行關鍵字過濾與 NLP 打分...")

filtered_data = []

# 2. 關鍵字過濾、13:30 時間切分與 NLP 打分
for news in raw_news:
    for code, name in TARGET_STOCKS.items():
        if name in news['title'] or code in news['title'] or name in news['summary']:

            content = news['title'] + " " + news['summary']
            sentiment = get_finbert_score(content)
            aligned_date = align_trading_date(news['publish_time'])

            filtered_data.append({
                'Date': aligned_date,
                'ticker': f"{code}.TW",
                'Daily_Sentiment': sentiment,
                'Original_Time': news['publish_time'].strftime('%Y-%m-%d %H:%M:%S'),
                'Title': news['title']
            })

if not filtered_data:
    print("⚠️ 今天沒有這五檔股票的相關新聞。")
else:
    df_today = pd.DataFrame(filtered_data)

    # 3. 聚合成每日單一分數 (同股票若有多則新聞則取平均)
    df_today_agg = df_today.groupby(['ticker', 'Date'])['Daily_Sentiment'].mean().reset_index()

    print("\n✅ 今日情緒特徵產出完成！(準備交接給 LSTM 模型)")
    display(df_today_agg)

    print("\n🔍 盤後驗證：查看新聞標題與對應的情緒分數、對齊日期")
    display(df_today[['ticker', 'Original_Time', 'Date', 'Daily_Sentiment', 'Title']].head(10))

    # 在實務上，你會在這裡把 df_today_agg 附加(Append)到 lstm_multivariate_features_2026.csv 中

🚀 開始執行每日增量結算任務: 2026-10-12 20:08:00
📡 成功抓取全市場今日新聞共 174 筆，開始進行關鍵字過濾與 NLP 打分...

✅ 今日情緒特徵產出完成！(準備交接給 LSTM 模型)


,ticker,Date,Daily_Sentiment
0,2002.TW,2026-10-12,0.958450
1,2317.TW,2026-10-12,0.382425
2,2330.TW,2026-10-12,0.544836
3,2603.TW,2026-10-12,0.806414



🔍 盤後驗證：查看新聞標題與對應的情緒分數、對齊日期


,ticker,Original_Time,Date,Daily_Sentiment,Title
0,2317.TW,2026-10-12 11:15:33,2026-10-12,-0.5171,鴻海處分臻鼎持股5544張 均價584.96元獲利26.83億元
1,2603.TW,2026-10-12 11:04:51,2026-10-12,0.9521,響應淨零政策 李長榮AquaNova以AI智能破解廢水用電排碳痛點
2,2603.TW,2026-10-12 10:29:27,2026-10-12,0.8553,【量大強漲股整理】【 3265 台星科】【 2340 台亞】【6742澤米】【3624光頡】...
3,2002.TW,2026-10-12 10:13:46,2026-10-12,0.9616,中鋼9月營收310億元創28個月新高 Q4迎傳統旺季鋼市逐步向好
4,2330.TW,2026-10-12 10:12:13,2026-10-12,0.9942,多頭根基穩固拉回就是機會、建榮、德宏、世芯、大立光⊕、天二科⊕、冠西電⊕
5,2002.TW,2026-10-12 09:52:12,2026-10-12,0.9553,營收速報 - 中鋼(2002)9月營收310.01億元年增率高達27.13％
6,2330.TW,2026-10-12 09:33:18,2026-10-12,0.9942,台積電利多將出！誰是下一匹黑馬？
7,2603.TW,2026-10-12 08:10:59,2026-10-12,0.8888,鉅亨速報 - Factset 最新調查：長榮(2603-TW)EPS預估上修至39.33元，...
8,2317.TW,2026-10-12 08:00:13,2026-10-12,0.9850,鴻海加碼投資越南！旗下富康科技資本額翻倍，年營收86億美元、三年暴增百倍
9,2330.TW,2026-10-12 07:30:14,2026-10-12,-0.1448,家碩(6953) 合約負債創近期新高！台積電N2放量與EUV光罩需求暴增！
